In [ ]:
# ==============================================================================
# CÉLULA 1: SETUP E DEFINIÇÃO DO DIRETÓRIO RAIZ (ROOT)
# ==============================================================================
import os
import sys

# 1. Verificação do ambiente de execução (Nuvem vs Local)
if 'google.colab' in sys.modules:
    print("☁️ Ambiente detectado: Google Colab")
    from google.colab import drive
    drive.mount('/content/drive')

    # Insira abaixo o caminho exato onde clonou/salvou este repositório no seu Google Drive.
    caminho_projeto = 'CAMINHO_COMPLETO/tcc_risco_queda_v-pub''

else:
    print("💻 Ambiente detectado: Local (Jupyter, VS Code, Spyder)")
    # Assume dinamicamente a pasta onde o notebook foi aberto como a raiz
    caminho_projeto = os.getcwd()

# 2. Configuração global do diretório de trabalho
try:
    os.chdir(caminho_projeto)
    if caminho_projeto not in sys.path:
        sys.path.append(caminho_projeto)
    print(f"✅ Diretório raiz configurado com sucesso: {os.getcwd()}")
except FileNotFoundError:
    print(f"❌ ERRO: A pasta '{caminho_projeto}' não foi encontrada.")
    print("Por favor, verifique se o caminho inserido está correto.")
    raise

☁️ Ambiente detectado: Google Colab
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Diretório raiz configurado com sucesso: /content/drive/MyDrive/1) PESQUISA/ESALQ Data Science/tcc/tema_classificacao_queda_arvore/git/tcc_risco_queda_v-pub


In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np

pd.set_option('display.max_columns', None)

if caminho_projeto not in sys.path:
    sys.path.append(caminho_projeto)

# 4. Importa as suas funções da pasta src
from src.spatial_features import (
    preparar_indice_espacial,
    calcular_idag,
    identificar_aiv,
    calcular_dva,
    calcular_icc,
    get_azimute
)

from src.ETL_functions import imputar_por_proximidade

In [ ]:
# ==========================================
# 2. CARREGAMENTO DOS DATASETS
# ==========================================
arvores = gpd.read_file('data/processed/A_arvores.gpkg').to_crs(epsg=31983)
vias_calcadas = gpd.read_file('data/processed/B_vias_calcadas.gpkg').to_crs(epsg=31983)
quedas = gpd.read_file('data/processed/C_quedas_2014_2017.gpkg').to_crs(epsg=31983)

# Renomear as colunas de vias_calcadas para o padrão do Dicionário de Dados
vias_calcadas = vias_calcadas.rename(columns={
    'cvc_codlog': 'segmento_id',
    'cc_lg_min_min': 'via_calcada_largura_min',
    'cc_dec_max_max': 'via_calcada_declinacao_max'
    # Se houver coluna de comprimento, ex: 'extensao_km': 'via_extensao_km'
})

In [ ]:
# TRATAMENTO DE NANS E IMPUTAÇÃO

#Imputação Agrupada (Mediana pelo Tipo de Logradouro: RUA, AVENIDA, PRAÇA, etc.)

# Calculando medianas agrupadas e preenchendo os NaNs
vias_calcadas['via_calcada_largura_min'] = vias_calcadas.groupby('cvc_tplogr')['via_calcada_largura_min'].transform(
    lambda x: x.fillna(x.median())
)
vias_calcadas['via_calcada_declinacao_max'] = vias_calcadas.groupby('cvc_tplogr')['via_calcada_declinacao_max'].transform(
    lambda x: x.fillna(x.median())
)

# 2.3 Fallback de Segurança
# Caso exista alguma categoria viária onde TUDO seja nulo, usamos a mediana global da cidade
mediana_global_largura = vias_calcadas['via_calcada_largura_min'].median()
mediana_global_inclinacao = vias_calcadas['via_calcada_declinacao_max'].median()

vias_calcadas['via_calcada_largura_min'] = vias_calcadas['via_calcada_largura_min'].fillna(mediana_global_largura)
vias_calcadas['via_calcada_declinacao_max'] = vias_calcadas['via_calcada_declinacao_max'].fillna(mediana_global_inclinacao)

# 2.4 Criação da feature de extensão viária (usada depois no ICC)
vias_calcadas['via_extensao_km'] = (vias_calcadas.geometry.length / 1000).astype('float32')

In [ ]:
# ==========================================
# 0. PREPARAÇÃO: O NOVO "ID GEOMÉTRICO"
# ==========================================
# Resetamos o index para garantir uma ordem limpa e criamos o spatial_uid
vias_calcadas = vias_calcadas.reset_index(drop=True)
vias_calcadas['spatial_uid'] = vias_calcadas.index

# ==========================================
# 1. ESCALA INDIVIDUAL: MÉTRICAS DAS ÁRVORES
# ==========================================
print("1. Calculando métricas individuais das árvores...")
coords, tree = preparar_indice_espacial(arvores)

arvores['idag_15_indiv'] = calcular_idag(coords, tree, raio_corte=15.0, beta=2.0)
arvores['is_isolada'] = identificar_aiv(coords, tree, limite_isolamento=15.0)
arvores['dva_3_indiv'] = calcular_dva(coords, tree, vizinhos=3)
arvores['idag_15_log_indiv'] = np.log1p(arvores['idag_15_indiv']).astype('float32')

# ==========================================
# 2. SJOIN: ÁRVORES -> VIAS (Garantia 1:1)
# ==========================================
print("2. Associando árvores às vias geometricamente...")
# O how='inner' remove árvores que estão no meio do nada (longe de vias)
arvores_com_via = gpd.sjoin_nearest(
    arvores,
    vias_calcadas[['spatial_uid', 'geometry']],
    how='inner',
    max_distance=20.0
)

# A TRAVA DO 1:1 -> Se uma árvore caiu a distâncias iguais de duas vias,
# o index dela ficará duplicado. Nós dropamos mantendo apenas a primeira via.
arvores_com_via = arvores_com_via[~arvores_com_via.index.duplicated(keep='first')]

# ==========================================
# 3. AGREGAÇÃO: ESCALA DA VIA (Versão Blindada)
# ==========================================
print("3. Agregando métricas arbóreas por via...")

# 3.1 Faz a agregação
df_agregado = arvores_com_via.groupby('spatial_uid').agg(
    via_arvores_contagem=('spatial_uid', 'count'),
    stat_aiv_arvores_isoladas=('is_isolada', 'sum'),
    stat_dva_vizinhanca_dist_med=('dva_3_indiv', 'mean'),
    stat_dva_vizinhanca_dist_std=('dva_3_indiv', 'std'),
    stat_idag_densidade_grav_med=('idag_15_log_indiv', 'mean'),
    stat_idag_densidade_grav_std=('idag_15_log_indiv', 'std')
).reset_index()

# 3.2 Preenchendo nulos de vias com apenas 1 árvore (onde o std dá NaN)
# A checagem 'in' evita que o código quebre se o df_agregado estiver vazio
if 'stat_dva_vizinhanca_dist_std' in df_agregado.columns:
    df_agregado['stat_dva_vizinhanca_dist_std'] = df_agregado['stat_dva_vizinhanca_dist_std'].fillna(0)
if 'stat_idag_densidade_grav_std' in df_agregado.columns:
    df_agregado['stat_idag_densidade_grav_std'] = df_agregado['stat_idag_densidade_grav_std'].fillna(0)

# 3.3 O PASSO CRÍTICO: Acoplando as agregações de volta na base principal
vias_calcadas = vias_calcadas.merge(df_agregado, on='spatial_uid', how='left')

# Lista de colunas alvo
cols_fill = [
    'via_arvores_contagem', 'stat_aiv_arvores_isoladas', 'stat_dva_vizinhanca_dist_med',
    'stat_dva_vizinhanca_dist_std', 'stat_idag_densidade_grav_med', 'stat_idag_densidade_grav_std'
]

# 3.4 Trava de segurança: Se por algum motivo as colunas não vieram no merge, nós as criamos zeradas
for col in cols_fill:
    if col not in vias_calcadas.columns:
        vias_calcadas[col] = 0.0

# Agora o fillna é 100% seguro
vias_calcadas[cols_fill] = vias_calcadas[cols_fill].fillna(0)

print("Agregação concluída com sucesso! ✅")

# ==========================================
# 4. INFRAESTRUTURA E CÁLCULOS GEOMÉTRICOS (ICC e Azimute)
# ==========================================
print("4. Calculando ICC e Azimute...")
vias_calcadas['via_extensao_km'] = (vias_calcadas.geometry.length / 1000).astype('float32')

# Calculando ICC (garanta que a coluna de largura não tenha NaNs neste momento)
vias_calcadas['via_icc_confinamento_idx'] = calcular_icc(
    df=vias_calcadas,
    col_arvores='via_arvores_contagem',
    col_largura='via_calcada_largura_min',
    col_comprimento_km='via_extensao_km'
).astype('float32')

# Azimute (usando a função otimizada para polígonos)
vias_calcadas['via_azimute_graus'] = vias_calcadas.geometry.apply(get_azimute).astype('float32')

# ==========================================
# 5. TARGET: QUEDAS -> VIAS (Garantia 1:1)
# ==========================================
print("5. Associando ocorrências de queda (Target)...")
quedas_com_via = gpd.sjoin_nearest(
    quedas,
    vias_calcadas[['spatial_uid', 'geometry']],
    how='inner',
    max_distance=50.0 # Tolerância maior para GPS de bombeiros/defesa civil
)

# A TRAVA DO 1:1 -> Uma mesma queda não pode contar para duas vias diferentes
quedas_com_via = quedas_com_via[~quedas_com_via.index.duplicated(keep='first')]

# Contando quantas quedas caíram em cada spatial_uid
agg_quedas = quedas_com_via.groupby('spatial_uid').size().reset_index(name='target_historico_quedas')

# Acoplando o target na base final
df_matriz_final = vias_calcadas.merge(agg_quedas, on='spatial_uid', how='left')
df_matriz_final['target_historico_quedas'] = df_matriz_final['target_historico_quedas'].fillna(0).astype('int64')
df_matriz_final['target_queda_bool'] = (df_matriz_final['target_historico_quedas'] > 0).astype('int64')

# Limpando o identificador temporário e o index
df_matriz_final = df_matriz_final.drop(columns=['spatial_uid'])

print(f"Pipeline concluído! Matriz final com {len(df_matriz_final)} linhas preservadas. ✅")

1. Calculando métricas individuais das árvores...
2. Associando árvores às vias geometricamente...
3. Agregando métricas arbóreas por via...
Agregação concluída com sucesso! ✅
4. Calculando ICC e Azimute...
5. Associando ocorrências de queda (Target)...
Pipeline concluído! Matriz final com 220064 linhas preservadas. ✅


In [ ]:
ordem_colunas = [
    # 1. Identificadores
    'cvc_nomelg',
    'cvc_tplogr',
    'cvc_classe',

    # 2. Infraestrutura
    'via_extensao_km',
    'via_azimute_graus',
    'via_calcada_largura_min',
    'via_calcada_declinacao_max',

    # 3. Árvores e Conflito
    'via_arvores_contagem',
    'via_icc_confinamento_idx',

    # 4. Métricas Espaciais
    'stat_aiv_arvores_isoladas',
    'stat_dva_vizinhanca_dist_med',
    'stat_dva_vizinhanca_dist_std',
    'stat_idag_densidade_grav_med',
    'stat_idag_densidade_grav_std',

    # 5. Geometria
    'geometry',

    # 6. Targets
    'target_historico_quedas',
    'target_queda_bool'
]

# Aplicando a reordenação no GeoDataFrame
df_matriz_final = df_matriz_final[ordem_colunas]

In [ ]:
# Selecionando apenas vias com árvores
df = df_matriz_final.copy()
df_fil = df[df['via_arvores_contagem'] > 0].reset_index(drop=True)

In [ ]:
# PERFIL DO DATASET

# ARVORES
df_fil['stat_aiv_arvores_isoladas'].sum() # 79597 isoladas
df_fil['via_arvores_contagem'].sum() # 651778 arvores

# QUEDAS
df_fil['target_historico_quedas'].sum() # 9530 quedas (2014-2017)

# VIAS
# 110957 segmentos de vias com árvores
len(df_fil[df_fil['via_arvores_contagem'] >= 1])
# 7326 segmentos de vias com ao menos 1 queda (2014-2017)
df_fil['target_queda_bool'].sum()

110957

In [ ]:
# Export
# df_fil.to_file('data/processed/dataset_vias_features_espaciais_DEDUP.gpkg', driver='gpkg')

# Juntando com socio, infra e geologica

In [ ]:
# merge features espaciais, socio, infra e geologicas
df_controle = pd.read_parquet('data/processed/01_dataset_controle.parquet')

In [ ]:
import geopandas as gpd

print("0. Decodificando geometrias WKB e criando GeoDataFrames...")

# 1. Converte o binário para geometria legível (se estiver em formato bytes)
if isinstance(df_controle['geometry'].iloc[0], bytes):
    df_controle['geometry'] = gpd.GeoSeries.from_wkb(df_controle['geometry'])

if isinstance(df_fil['geometry'].iloc[0], bytes):
    df_fil['geometry'] = gpd.GeoSeries.from_wkb(df_fil['geometry'])

df_controle = gpd.GeoDataFrame(df_controle, geometry='geometry', crs="EPSG:31983")
df_fil = gpd.GeoDataFrame(df_fil, geometry='geometry', crs="EPSG:31983")

print("1. Deduplicando a base df_controle...")
df_controle['geom_wkb'] = df_controle.geometry.apply(lambda geom: geom.wkb if geom else None)

df_controle_dedup = df_controle.drop_duplicates(subset=['geom_wkb'], keep='first').copy()
df_controle_dedup = df_controle_dedup.drop(columns=['geom_wkb'])

print(f"Linhas na df_controle antes: {len(df_controle)} | Após dedup: {len(df_controle_dedup)}")

print("2. Cruzando as bases com tolerância de precisão (0.1 metros)...")

# O geopandas tratará de cruzar todas as features automaticamente.
df_fil_merged = gpd.sjoin_nearest(
    df_fil,
    df_controle_dedup,
    how='left',
    max_distance=0.1,
    distance_col='dist_micrometrica'
)

# 3. Trava de segurança 1:1
# Remove duplicatas caso o sjoin_nearest encontre mais de um vizinho no raio de 10cm
df_fil_merged = df_fil_merged[~df_fil_merged.index.duplicated(keep='first')]

print(f"Merge concluído! Linhas preservadas: {len(df_fil_merged)}")

# Exibe as colunas finais para conferência
print(f"Colunas resultantes após o cruzamento espacial: {df_fil_merged.columns.tolist()}")

0. Decodificando geometrias WKB e criando GeoDataFrames...
1. Deduplicando a base df_controle...
Linhas na df_controle antes: 108226 | Após dedup: 107718
2. Cruzando as bases com tolerância de precisão (0.1 metros)...
Merge concluído! Linhas preservadas: 110957
Colunas resultantes após o cruzamento espacial: ['cvc_nomelg', 'cvc_tplogr', 'cvc_classe', 'via_extensao_km', 'via_azimute_graus', 'via_calcada_largura_min', 'via_calcada_declinacao_max', 'via_arvores_contagem', 'via_icc_confinamento_idx', 'stat_aiv_arvores_isoladas', 'stat_dva_vizinhanca_dist_med', 'stat_dva_vizinhanca_dist_std', 'stat_idag_densidade_grav_med', 'stat_idag_densidade_grav_std', 'geometry', 'target_historico_quedas', 'target_queda_bool', 'index_right', 'socio_zona_fiscal_cat', 'socio_vulnerabilidade_idx', 'socio_densidade_demog_hab_ha', 'infra_escolas_contagem', 'infra_semaforos_contagem', 'infra_iluminacao_contagem', 'infra_onibus_pontos_contagem', 'infra_uso_solo_cat', 'geo_declividade_terreno_cat', 'geo_relevo_

In [ ]:
# Imputando por proximidade

# Lista de colunas que seu .info() mostrou que têm nulos
colunas_vazias = [
    'socio_zona_fiscal_cat',
    'socio_vulnerabilidade_idx',
    'socio_densidade_demog_hab_ha',
    'geo_relevo_tipo_cat',
    'geo_litologia_tipo_cat'
]

# Executa a função
gdf_limpo = imputar_por_proximidade(df_fil_merged, colunas_vazias)

Imputando 1068 valores na coluna: socio_zona_fiscal_cat...
Imputando 1068 valores na coluna: socio_vulnerabilidade_idx...
Imputando 1068 valores na coluna: socio_densidade_demog_hab_ha...
Imputando 14578 valores na coluna: geo_relevo_tipo_cat...
Imputando 14578 valores na coluna: geo_litologia_tipo_cat...


In [ ]:
# Lista de colunas organizada por grupos temáticos
colunas_ordenadas = [
    # 1. IDENTIFICAÇÃO (Quem e onde é o segmento)
    'cvc_nomelg',
    'cvc_tplogr',
    'cvc_classe',

    # 2. CARACTERÍSTICAS DA VIA (Geometria viária e calçada)
    'via_extensao_km',
    'via_azimute_graus',
    'via_calcada_largura_min',
    'via_calcada_declinacao_max',

    # 3. VEGETAÇÃO E ESTATÍSTICAS ESPACIAIS (O objeto de estudo principal)
    'via_arvores_contagem',
    'via_icc_confinamento_idx',
    'stat_aiv_arvores_isoladas',
    'stat_dva_vizinhanca_dist_med',
    'stat_dva_vizinhanca_dist_std',
    'stat_idag_densidade_grav_med',
    'stat_idag_densidade_grav_std',

    # 4. CONTEXTO SOCIOECONÔMICO (Fatores humanos)
    'socio_zona_fiscal_cat',
    'socio_vulnerabilidade_idx',
    'socio_densidade_demog_hab_ha',

    # 5. INFRAESTRUTURA URBANA (Fatores de entorno)
    'infra_uso_solo_cat',
    'infra_escolas_contagem',
    'infra_semaforos_contagem',
    'infra_iluminacao_contagem',
    'infra_onibus_pontos_contagem',

    # 6. GEOGRAFIA E GEOLOGIA (Fatores de solo e relevo)
    'geo_declividade_terreno_cat',
    'geo_relevo_tipo_cat',
    'geo_litologia_tipo_cat',
    'geo_solo_mole_area_m2',

    # 7. TARGETS (O que o modelo deve prever)
    'target_historico_quedas',
    'target_queda_bool',

    # 8. GEOMETRIA (Sempre por último por convenção do GeoPandas)
    'geometry'
]

# Aplicando a nova ordem ao GeoDataFrame
gdf_limpo = gdf_limpo[colunas_ordenadas]

In [ ]:
# PERFIL DO DATASET

# ARVORES
gdf_limpo['stat_aiv_arvores_isoladas'].sum() # 79597 isoladas
gdf_limpo['via_arvores_contagem'].sum() # 651778 arvores

# QUEDAS
gdf_limpo['target_historico_quedas'].sum() # 9530 quedas (2014-2017)

# VIAS
# 110957 segmentos de vias com árvores
len(gdf_limpo[gdf_limpo['via_arvores_contagem'] >= 1])
# 7326 segmentos de vias com ao menos 1 queda (2014-2017)
gdf_limpo['target_queda_bool'].sum()

np.int64(7326)

In [ ]:
#gdf_limpo.to_file('data/processed/dataset_all_features_imputed_DEDUP.gpkg', driver='gpkg')